# Exercise 3

Consider Heston's stochastic volatility model:

\begin{cases}
dS_t = \mu S_t dt + \sqrt{\nu_t} S_t dB_t^1,\\
d\nu_t = k(\theta - \nu_t) dt + \eta \sqrt{\nu_t} dB_t^2,
\end{cases}

where $B^1$ and $B^2$ are correlated Brownian motions, defined as:

\begin{cases} 
B_t^1 =\sqrt{1-\rho^2}W_t^1 + \rho W_t^2, \\
B_t^2 = W_t^2,
\end{cases}

where $W^1$ and $W^2$ are standard Brownian motions. The dynamics of the underlying asset are expressed under the martingale measure $\mathbb{Q}$, therefore $\mu$ represents the market risk-free rate.

In [347]:
# Parameters of the Heston stochastic volatility model:

S0 = 1.0        # initial price of the underlying asset
mu = 0.03       # drift of the underlying asset, corresponds to the risk-free rate
k = 5.0         # speed of mean reversion of the variance
v0 = 0.04       # initial variance
theta = 0.04    # long-term variance
eta = 0.3       # volatility of the variance
rho = -0.7      # correlation between the two Brownian motions B^1, B^2
T = 0.5         # maturity in years

# array of Call option strikes
KK = np.arange(start=0.5, stop=1.5 + 1e-10, step=0.1)

This choice of parameters satisfies the Feller condition, since:

In [204]:
2*k*theta > eta**2

True

Thus, the solution to the volatility SDE remains positive and is well-defined:
$$
\nu_t > 0\quad \text{for all }t\geq 0, \quad\text{almost surely.}
$$
We write the dynamics of the log-price of the underlying asset. Letting $X_t=log(S_t)$, by It$\hat{o}$'s formula we have:
\begin{align}
dX_t &= \frac{1}{S_t} dS_t - \frac{1}{2S_t^2}d\braket{S}_t\\
&= \mu dt + \sqrt{\nu_t}dB_t^1 - \frac{1}{2S_t^2} \nu_t S_t^2 d\braket{B^1}_t \\
&= \mu dt + \sqrt{\nu_t}\left(\sqrt{1-\rho^2} dW_t^1 + \rho dW_t^2\right) - \frac{1}{2} \nu_t d\braket{\sqrt{1-\rho^2} W^1+\rho W^2}_t \\
&= \mu dt + \sqrt{(1-\rho^2)\nu_t} dW_t^1 + \rho \sqrt{\nu_t} dW_t^2 - \frac{1}{2} \nu_t \left( (1-\rho^2)dt + \rho^2 dt\right) \\
&= \left(\mu - \frac{1}{2} \nu_t \right) dt + \sqrt{(1-\rho^2)\nu_t} dW_t^1 + \rho \sqrt{\nu_t} dW_t^2  \\ 
\end{align}
By simulating the log-price dynamics, we transform the initial SDE into a new SDE where the coefficients do not depend on the price, thereby reducing the approximation error and making the method more stable.
We write the Euler discretization for the log-price and the volatility:
$$
\begin{cases}
X_{t+\Delta_t} = X_t + \left(\mu - \frac{1}{2} \nu_t \right) \Delta_t + \sqrt{(1-\rho^2)\nu_t} \Delta W_t^1 + \rho \sqrt{\nu_t} \Delta W_t^2, \quad\quad\quad\Delta W_t^1,\Delta W_t^2\sim\mathcal{N}(0,\Delta_t),\\
\nu_{t+\Delta_t} = \nu_t + k(\theta - \nu_t) \Delta_t + \eta \sqrt{\nu_t} \Delta W_t^2,
\end{cases}
$$
Even though the Feller condition is satisfied, when we discretize the process it is possible for $\nu_t$ to become negative, causing issues when evaluating the square root. For this reason, we implement the following control on volatility, known as the Full Truncation Scheme, which is one of the best-performing methods according to the literature:
$$
\begin{cases}
\nu_{pos} = \max{\{\nu_t, 0\}} \\
X_{t+\Delta_t} = X_t + \left(\mu - \frac{1}{2} \nu_{pos} \right) \Delta_t + \sqrt{(1-\rho^2)\nu_{pos}} \Delta W_t^1 + \rho \sqrt{\nu_{pos}} \Delta W_t^2, \\
\nu_{t+\Delta_t} = \nu_t + k(\theta - \nu_{pos}) \Delta_t + \eta \sqrt{\nu_{pos}} \Delta W_t^2,
\end{cases}
$$
With this solution, we set the variance value to zero whenever it becomes negative before proceeding to the calculation of the next time step.

### Code

In [208]:
import numpy as np
from scipy import stats
from scipy import linalg
from scipy import optimize
import matplotlib.pyplot as plt

In [552]:
# vectorized version for European Call option pricing
# K can be an array

def heston_call(S0, mu, k, v0, theta, eta, rho, K, T, m, dt):
    tt = np.arange(start=0, stop=T + 1e-10, step=dt)
    n = len(tt)
    
    X0 = np.log(S0)
    X = np.zeros(shape=(n,m))
    X[0,:] = X0
    
    v = np.zeros(shape=(n,m))
    v[0,:] = v0
    
    for i in range(n-1):
        dW1 = stats.norm.rvs(size=m) * np.sqrt(dt)
        dW2 = stats.norm.rvs(size=m) * np.sqrt(dt)

        # implementing the Full Truncation Scheme for negative variance
        v_pos = np.maximum(v[i,:], 0)
        X[i+1,:] = X[i,:] + (mu-0.5*v_pos)*dt + np.sqrt((1-rho**2)*v_pos)*dW1 + rho*np.sqrt(v_pos)*dW2
        v[i+1,:] = v[i,:] + k*(theta - v_pos)*dt + eta*np.sqrt(v_pos)*dW2
    
    S = np.exp(X)

    # adjusting the shape of K
    K_array = np.asarray(K)
    if K_array.ndim > 0:
        # transforming K into a column vector
        K_array = K_array[:, np.newaxis]
        
    payoff = np.exp(-np.multiply(mu,T)) * (S[-1,:] - K_array).clip(min=0)
    
    # to compute the expected value we calculate the sample mean
    # of the payoffs along the last axis (of size m)
    # since np.shape(payoff) = (K_array.ndim, m)
    return {
        'price': payoff.mean(axis=-1),  
        'S': S,
        'X': X,
        'v': v,
    }

### Plot of the underlying paths in the Heston model

In [559]:
n_sim = 500
tt = np.arange(start=0, stop=T + 1e-10, step=dt)

results = heston_call(S0, mu, k, v0, theta, eta, rho, KK, T, n_sim, dt)
S_heston = results['S']
X_heston = results['X']
v_heston = results['v']